# 노트북 3. Single-turn vs Multi-turn + 대화 저장 전략

> Phase 1 — 기초: LLM과 대화하는 법

LLM에는 '기억'이 없습니다.
매번 전체 대화 내역을 통째로 보내야 합니다.
그 대화를 어디에 저장하느냐가 아키텍처 결정입니다.

**학습 목표**
- Single-turn 호출의 한계를 이해한다
- Multi-turn 대화의 작동 원리(messages 리스트 누적)를 설명할 수 있다
- google-genai와 LangChain에서 멀티턴 대화를 구현할 수 있다
- 대화 저장소(InMemory / SQLite)의 특성과 선택 기준을 이해한다
- LangGraph의 MessagesState를 활용한 상태 관리 패턴을 익힌다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | Single-turn 한계 + Multi-turn 원리 + 저장소 비교 + LangGraph | 읽고 실행 |
| Part 2 — 실습 | google-genai 멀티턴 + LangChain 히스토리 + LangGraph Saver | 직접 작성 |
| Part 3 — 챌린지 | 대화 맥락을 유지하는 Q&A 봇 구현 | 강사와 함께 |

In [32]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai langgraph langgraph-checkpoint-sqlite

import os
from google import genai

# API 키 설정 (Colab 환경 기준)
# 왼쪽 키 아이콘 → GEMINI_API_KEY 등록 후 실행
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

# LangChain 모델도 미리 생성
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_API_KEY,
)

print("환경 설정 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.6/151.6 kB 6.4 MB/s eta 0:00:00
환경 설정 완료


---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 Single-turn의 한계

LLM API 호출은 본질적으로 **stateless**(상태 없음)입니다.
매 호출은 독립적이며, 이전 호출의 내용을 전혀 기억하지 못합니다.

이것을 **Single-turn**(단일 턴) 호출이라 합니다.
한 번 질문하고 한 번 답변을 받으면 끝입니다.

아래 코드는 이 한계를 직접 확인합니다.
첫 번째 호출에서 이름을 알려주고, 두 번째 호출에서 이름을 물어봅니다.

In [2]:
# Single-turn의 한계 — 이전 대화를 기억하지 못함

# 첫 번째 호출: 이름을 알려줌
resp1 = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="제 이름은 김철수입니다. 반갑습니다!",
)
print(f"[1번째 호출] {resp1.text}")
print()

[1번째 호출] 김철수님, 반갑습니다! 저는 인공지능 챗봇입니다. 무엇을 도와드릴까요?



In [3]:
# 두 번째 호출: 이름을 물어봄 — 기억하지 못함
resp2 = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="제 이름이 뭐라고 했죠?",
)
print(f"[2번째 호출] {resp2.text}")

[2번째 호출] 저는 사용자님의 이름을 알지 못합니다. 저는 개인 정보를 저장하거나 이전 대화 내용을 기억하지 않기 때문입니다.

혹시 저에게 이름을 알려주시면, 지금 대화 중에 참고하여 불러드릴 수 있습니다!


모델은 "제 이름이 뭐라고 했죠?"라는 질문에 답하지 못합니다.
두 번째 호출은 첫 번째 호출과 완전히 별개의 요청이기 때문입니다.

```
호출 1: "제 이름은 김철수입니다"  →  "반갑습니다, 김철수님!"     (독립 요청)
호출 2: "제 이름이 뭐라고 했죠?"  →  "이름을 알려주시겠어요?"   (독립 요청)
```

> 핵심: LLM에는 '기억'이 없습니다.
> "아까 말한 거"라고 하면 "무엇을 말씀하셨는지 모르겠습니다"가 나옵니다.
> 이것이 Single-turn의 본질적 한계입니다.

LangChain에서도 마찬가지입니다.
`.invoke()`는 매번 독립적인 호출입니다.

In [4]:
# LangChain에서도 동일한 한계
resp_lc1 = model.invoke("제 이름은 김철수입니다. 반갑습니다!")
print(f"[1번째] {resp_lc1.content}")
print()

resp_lc2 = model.invoke("제 이름이 뭐라고 했죠?")
print(f"[2번째] {resp_lc2.content}")

[1번째] 네, 김철수님, 반갑습니다! 저도 만나뵙게 되어 기쁩니다.
무엇을 도와드릴까요?

[2번째] 죄송합니다만, 저는 사용자님의 이름을 따로 저장하거나 기억하고 있지 않습니다.

저는 인공지능이기 때문에 이전 대화 내용을 개별적으로 기억하거나 사용자 정보를 저장하지 않습니다.

혹시 이름을 알려주시면 제가 이번 대화 안에서 사용자님을 부르거나 참고하는 데 사용할 수 있습니다! 😊


## 1.2 Multi-turn의 작동 원리

**Multi-turn**(멀티턴) 대화의 핵심 원리는 단순합니다:
클라이언트(개발자 코드)에서 **messages 리스트를 누적 관리**하고,
매 호출 시 **전체 대화 이력을 통째로 전송**하는 것입니다.

```
턴 1 전송: [user: "이름은 김철수"]
턴 1 응답:               → [assistant: "반갑습니다"]

턴 2 전송: [user: "이름은 김철수", assistant: "반갑습니다", user: "내 이름이 뭐였죠?"]
턴 2 응답:               → [assistant: "김철수님이시죠"]
```

모델이 기억하는 것이 아닙니다.
**우리가 매번 전체 대화를 다시 보내주는 것**입니다.

> 핵심: 멀티턴 대화는 모델의 능력이 아니라 클라이언트의 책임입니다.
> 모델은 받은 메시지 리스트를 보고 "하나의 긴 대화"로 이해할 뿐입니다.

### 가장 단순한 멀티턴: 리스트 수동 관리

가장 기본적인 방법은 파이썬 리스트로 메시지를 직접 관리하는 것입니다.

아래 코드는 LangChain 메시지 리스트를 수동으로 관리하여 멀티턴 대화를 구현합니다.

In [5]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# 대화 이력을 담을 리스트
conversation = [
    SystemMessage(content="당신은 친절한 도우미입니다. 간결하게 답변하세요."),
]

# 턴 1: 이름 알려주기
conversation.append(HumanMessage(content="제 이름은 김철수입니다."))
resp1 = model.invoke(conversation)
conversation.append(resp1)  # AI 응답도 이력에 추가

print(f"[턴 1] {resp1.content}")
print(f"현재 메시지 수: {len(conversation)}개")
print()

[턴 1] 안녕하세요, 김철수님!
현재 메시지 수: 3개



In [6]:
# 턴 2: 이름 물어보기 — 이번에는 전체 이력을 함께 전송
conversation.append(HumanMessage(content="제 이름이 뭐라고 했죠?"))
resp2 = model.invoke(conversation)
conversation.append(resp2)

print(f"[턴 2] {resp2.content}")
print(f"현재 메시지 수: {len(conversation)}개")

[턴 2] 김철수님입니다.
현재 메시지 수: 5개


이번에는 모델이 이름을 기억합니다.
턴 2에서 전체 대화 이력(system + 턴 1 + 턴 2)을 함께 전송했기 때문입니다.

현재 대화 이력의 구조를 확인해봅시다.

In [7]:
# 대화 이력 구조 확인
for i, msg in enumerate(conversation):
    role = msg.type  # 'system', 'human', 'ai'
    content = msg.content[:50]  # 앞 50자만
    print(f"  [{i}] {role:6s} | {content}")

  [0] system | 당신은 친절한 도우미입니다. 간결하게 답변하세요.
  [1] human  | 제 이름은 김철수입니다.
  [2] ai     | 안녕하세요, 김철수님!
  [3] human  | 제 이름이 뭐라고 했죠?
  [4] ai     | 김철수님입니다.


### 멀티턴의 비용 문제

멀티턴 대화에서는 매 호출마다 **전체 이력을 전송**합니다.
대화가 길어질수록 입력 토큰 수가 누적됩니다.

```
턴 1: system(50) + user(20)                     = 70 토큰
턴 2: system(50) + user(20) + ai(30) + user(15) = 115 토큰
턴 3: 위 전체 + ai(25) + user(20)               = 160 토큰
...
턴 20: 수천 토큰
```

이 문제를 해결하는 전략은 노트북 7(컨텍스트 매니지먼트)에서 다룹니다.

아래 코드는 대화를 이어가면서 토큰 사용량이 어떻게 증가하는지 확인합니다.

In [8]:
# 멀티턴에서 토큰 누적 확인
conv = [SystemMessage(content="간결하게 답변하세요.")]

questions = [
    "파이썬이란?",
    "자바와의 차이점은?",
    "어떤 것을 먼저 배우는 게 좋을까요?",
    "취업에는 어떤 언어가 유리한가요?",
]

for i, q in enumerate(questions, 1):
    conv.append(HumanMessage(content=q))
    resp = model.invoke(conv)
    conv.append(resp)

    tokens = resp.usage_metadata
    print(f"턴 {i}: 입력 {tokens['input_tokens']}토큰 / 출력 {tokens['output_tokens']}토큰 / 메시지 {len(conv)}개")

턴 1: 입력 12토큰 / 출력 918토큰 / 메시지 3개
턴 2: 입력 66토큰 / 출력 1209토큰 / 메시지 5개
턴 3: 입력 389토큰 / 출력 1381토큰 / 메시지 7개
턴 4: 입력 678토큰 / 출력 1297토큰 / 메시지 9개


턴이 진행될수록 입력 토큰 수가 증가하는 것을 확인할 수 있습니다.
이전 대화 전체가 매번 입력으로 들어가기 때문입니다.

## 1.3 google-genai에서의 멀티턴

google-genai SDK에서 멀티턴 대화를 구현하는 방법은 두 가지입니다:

1. **contents 리스트 수동 관리**: `user`/`model` role을 교대로 쌓기
2. **client.chats.create()**: SDK가 제공하는 세션 객체 활용

### 방법 1: contents 리스트 수동 관리

google-genai에서는 `contents`에 `Content` 객체 리스트를 전달합니다.
각 메시지에 `role="user"` 또는 `role="model"`을 지정합니다.

아래 코드는 contents 리스트를 수동으로 관리하는 멀티턴 대화를 보여줍니다.

In [9]:
from google.genai.types import Content, Part

# contents 리스트를 직접 관리
contents = []

# 턴 1
contents.append(Content(role="user", parts=[Part(text="제 이름은 이영희입니다.")]))

resp1 = client.models.generate_content(
    model="gemini-2.5-flash",
    config={"system_instruction": "친절하게 답변하세요."},
    contents=contents,
)

# 모델 응답도 contents에 추가
contents.append(Content(role="model", parts=[Part(text=resp1.text)]))
print(f"[턴 1] {resp1.text}")
print()

[턴 1] 네, 이영희 님이시군요! 반갑습니다! 😊



In [10]:
# 턴 2
contents.append(Content(role="user", parts=[Part(text="제 이름이 뭐였죠?")]))

resp2 = client.models.generate_content(
    model="gemini-2.5-flash",
    config={"system_instruction": "친절하게 답변하세요."},
    contents=contents,
)

contents.append(Content(role="model", parts=[Part(text=resp2.text)]))
print(f"[턴 2] {resp2.text}")
print(f"\ncontents 길이: {len(contents)}개")

[턴 2] 네, 물론이죠! **이영희** 님이십니다. 😊 제가 기억하고 있습니다.

contents 길이: 4개


### 방법 2: client.chats.create() 세션

google-genai SDK는 `client.chats.create()`로 **채팅 세션**을 생성할 수 있습니다.
세션 객체가 내부적으로 대화 이력을 관리해주므로, 개발자가 직접 리스트를 관리할 필요가 없습니다.

아래 코드는 채팅 세션을 사용한 멀티턴 대화를 보여줍니다.

In [11]:
# 채팅 세션 생성
chat = client.chats.create(
    model="gemini-2.5-flash",
    config={"system_instruction": "친절하게 답변하세요."},
)

# 턴 1
resp1 = chat.send_message("제 이름은 박민수입니다.")
print(f"[턴 1] {resp1.text}")
print()

[턴 1] 네, 박민수 님이시군요! 말씀해주셔서 감사합니다. 만나 뵙게 되어 반갑습니다. 😊



In [12]:
# 턴 2 — 세션이 이전 대화를 자동으로 포함
resp2 = chat.send_message("제 이름이 뭐였죠?")
print(f"[턴 2] {resp2.text}")
print()

[턴 2] 네, 박민수 님이십니다! 😊



In [13]:
# 턴 3 — 이전 맥락이 계속 유지됨
resp3 = chat.send_message("제 이름의 성(姓)은 뭔가요?")
print(f"[턴 3] {resp3.text}")

[턴 3] 박민수 님의 성(姓)은 **박** 입니다.


세션 내부에 쌓인 대화 이력을 확인해봅시다.

In [14]:
# 세션에 쌓인 대화 이력 확인
# chat._curated_history에 전체 이력이 저장되어 있음
for i, msg in enumerate(chat._curated_history):
    role = msg.role
    text = msg.parts[0].text[:60]
    print(f"  [{i}] {role:5s} | {text}")

  [0] user  | 제 이름은 박민수입니다.
  [1] model | 네, 박민수 님이시군요! 말씀해주셔서 감사합니다. 만나 뵙게 되어 반갑습니다. 😊
  [2] user  | 제 이름이 뭐였죠?
  [3] model | 네, 박민수 님이십니다! 😊
  [4] user  | 제 이름의 성(姓)은 뭔가요?
  [5] model | 박민수 님의 성(姓)은 **박** 입니다.


> 핵심: `client.chats.create()`는 편리하지만, 내부적으로 하는 일은 동일합니다.
> 대화 이력을 리스트에 쌓고, 매 호출 시 전체를 전송합니다.
> 프로세스가 종료되면 세션도 사라집니다 (InMemory).

## 1.4 LangChain에서의 멀티턴

LangChain에서 멀티턴 대화를 관리하는 방법은 크게 두 가지입니다:

1. **List[BaseMessage] 수동 관리**: 앞서 본 것처럼 리스트에 직접 추가
2. **ChatMessageHistory**: LangChain이 제공하는 이력 관리 클래스

### ChatMessageHistory

`InMemoryChatMessageHistory`는 대화 이력을 메모리에 저장하는 클래스입니다.
이력 관리에 필요한 메서드(`add_message`, `clear` 등)를 제공합니다.

아래 코드는 `InMemoryChatMessageHistory`를 사용한 대화 관리를 보여줍니다.

In [15]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# 대화 이력 저장소 생성
history = InMemoryChatMessageHistory()

# system 메시지 추가
history.add_message(SystemMessage(content="당신은 친절한 도우미입니다. 간결하게 답변하세요."))

# 턴 1
user_input = "제 이름은 정수진입니다."
history.add_message(HumanMessage(content=user_input))

resp = model.invoke(history.messages)
history.add_message(resp)

print(f"[턴 1] {resp.content}")
print(f"이력 수: {len(history.messages)}")
print()

[턴 1] 안녕하세요, 정수진 님.
이력 수: 3



In [16]:
# 턴 2
user_input = "제 이름이 뭐였죠?"
history.add_message(HumanMessage(content=user_input))

resp = model.invoke(history.messages)
history.add_message(resp)

print(f"[턴 2] {resp.content}")
print(f"이력 수: {len(history.messages)}")

[턴 2] 정수진 님이십니다.
이력 수: 5


In [17]:
# 저장된 이력 확인
for i, msg in enumerate(history.messages):
    print(f"  [{i}] {msg.type:6s} | {msg.content[:60]}")

  [0] system | 당신은 친절한 도우미입니다. 간결하게 답변하세요.
  [1] human  | 제 이름은 정수진입니다.
  [2] ai     | 안녕하세요, 정수진 님.
  [3] human  | 제 이름이 뭐였죠?
  [4] ai     | 정수진 님이십니다.


### 대화 함수로 감싸기

매번 `history.add_message()`를 호출하는 것은 번거롭습니다.
이를 함수로 감싸면 대화 루프를 깔끔하게 만들 수 있습니다.

아래 코드는 대화 함수를 만들어 여러 턴을 처리하는 패턴을 보여줍니다.

In [18]:
def chat_turn(user_input: str, history: InMemoryChatMessageHistory) -> str:
    """한 턴의 대화를 처리하고 응답을 반환"""
    history.add_message(HumanMessage(content=user_input))
    response = model.invoke(history.messages)
    history.add_message(response)
    return response.content


# 새 대화 시작
my_history = InMemoryChatMessageHistory()
my_history.add_message(SystemMessage(content="당신은 여행 가이드입니다. 간결하게 답변하세요."))

# 여러 턴 대화
questions = [
    "서울에서 가볼 만한 곳을 추천해주세요.",
    "그 중에서 맛집이 많은 곳은?",
    "거기서 유명한 음식은 뭔가요?",
]

for i, q in enumerate(questions, 1):
    answer = chat_turn(q, my_history)
    print(f"[턴 {i}] Q: {q}")
    print(f"         A: {answer}")
    print()

[턴 1] Q: 서울에서 가볼 만한 곳을 추천해주세요.
         A: 네, 서울에서 가볼 만한 곳을 추천해 드립니다.

*   **경복궁:** 한국의 대표적인 궁궐.
*   **북촌 한옥마을:** 전통 가옥이 아름다운 곳.
*   **명동:** 쇼핑과 길거리 음식을 즐기기 좋은 번화가.
*   **N서울타워:** 서울 전경을 감상할 수 있는 랜드마크.
*   **홍대:** 젊음과 예술의 거리.
*   **인사동:** 전통 문화와 공예품을 만날 수 있는 곳.
*   **광장시장:** 다양한 먹거리를 맛볼 수 있는 전통 시장.

[턴 2] Q: 그 중에서 맛집이 많은 곳은?
         A: 네, 맛집이 많은 곳은 다음과 같습니다:

*   **명동:** 다양한 종류의 식당과 길거리 음식이 많습니다.
*   **홍대:** 젊은 감각의 트렌디한 식당과 카페가 많습니다.
*   **광장시장:** 전통적인 시장 음식(빈대떡, 마약김밥 등)을 맛볼 수 있습니다.

[턴 3] Q: 거기서 유명한 음식은 뭔가요?
         A: 네, 각 지역에서 유명한 음식은 다음과 같습니다:

*   **명동:**
    *   **명동교자 칼국수:** 쫄깃한 면발과 진한 육수가 특징인 칼국수.
    *   **길거리 음식:** 떡볶이, 계란빵, 닭꼬치 등 다양한 길거리 간식.
    *   **닭갈비:** 매콤한 양념에 볶아 먹는 닭고기 요리.

*   **홍대:**
    *   **치맥 (치킨과 맥주):** 다양한 프랜차이즈 및 개성 있는 치킨집.
    *   **퓨전 음식:** 트렌디하고 독창적인 파스타, 버거, 일식 등.
    *   **카페 디저트:** 개성 있는 카페에서 즐기는 케이크, 빵, 음료.

*   **광장시장:**
    *   **빈대떡:** 녹두를 갈아 바삭하게 부쳐낸 전.
    *   **마약김밥:** 중독성 있는 맛의 미니 김밥.
    *   **육회:** 신선한 소고기를 양념에 버무린 요리.



## 1.5 대화 저장소 비교

지금까지의 모든 방법은 **InMemory**(메모리 내) 저장 방식입니다.
프로세스가 종료되면 대화 이력이 사라집니다.

실제 서비스에서는 대화를 영구적으로 저장해야 할 수 있습니다.
저장소 선택은 다음 기준으로 판단합니다:

| 저장소 | 영속성 | 속도 | 용도 |
|--------|--------|------|------|
| InMemory | 프로세스 종료 시 소멸 | 가장 빠름 | 프로토타입, 테스트 |
| SQLite | 파일로 영구 저장 | 빠름 | 단일 서버, 소규모 |
| Redis | TTL로 자동 만료 가능 | 매우 빠름 | 다중 서버, 세션 관리 |
| PostgreSQL | 영구 보존 + 검색/분석 | 보통 | 운영 환경, 대규모 |

> 선택 기준: 대화를 얼마나 오래 보관해야 하는가?
> 서버가 몇 대인가? 대화 이력을 나중에 분석해야 하는가?

이 노트북에서는 InMemory와 SQLite 두 가지를 다룹니다.

### SQLite 저장소 확인

LangGraph의 `SqliteSaver`를 사용하면 대화 이력을 SQLite 파일에 저장할 수 있습니다.
이 부분은 다음 섹션(1.6 LangGraph)에서 함께 다룹니다.

## 1.6 LangGraph의 MessagesState와 add_messages

**LangGraph**는 LangChain 팀이 만든 에이전트 프레임워크입니다.
그래프 기반으로 LLM 애플리케이션의 흐름을 정의합니다.

LangGraph에서 멀티턴 대화를 관리하는 핵심 개념:

1. **MessagesState**: 메시지 리스트를 상태로 관리하는 미리 정의된 타입
2. **add_messages**: 새 메시지를 기존 이력에 추가하는 **reducer**(리듀서)
3. **Checkpointer**: 상태를 저장하는 백엔드 (MemorySaver, SqliteSaver 등)

LangGraph를 사용하면 대화 관리를 그래프의 상태(state)로 다룰 수 있습니다.
이후 노트북 11에서 이 패턴을 본격적으로 활용합니다.

> **reducer**(리듀서)란: 새로운 값이 들어왔을 때 기존 상태와 어떻게 합칠지를 결정하는 함수입니다.
> `add_messages`는 "새 메시지를 기존 리스트에 append"하는 reducer입니다.

### 기본 구조: StateGraph + MessagesState

아래 코드는 LangGraph로 가장 간단한 대화 그래프를 구성합니다.
노드 하나에서 LLM을 호출하는 최소 구조입니다.

```python
from typing import Annotated
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

In [26]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState



# 노드 함수: LLM을 호출하고 응답을 상태에 추가
def chat_node(state: MessagesState):
    """메시지 이력을 받아 LLM을 호출하고 응답을 반환"""
    response = model.invoke(state["messages"])
    # {"messages": [response]}를 반환하면
    # add_messages reducer가 기존 이력에 append함
    return {"messages": [response]}


# 그래프 정의: START → chat_node → END
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("chat", chat_node)
graph_builder.add_edge(START, "chat")
graph_builder.add_edge("chat", END)

print("그래프 구조 정의 완료")
print("START → chat → END")

그래프 구조 정의 완료
START → chat → END


### MemorySaver로 상태 유지

그래프를 컴파일할 때 **Checkpointer**를 지정하면, 호출 간에 상태가 유지됩니다.
`MemorySaver`는 메모리에 상태를 저장하는 가장 간단한 checkpointer입니다.

아래 코드는 MemorySaver를 사용한 멀티턴 대화를 보여줍니다.

In [27]:
from langgraph.checkpoint.memory import MemorySaver

# MemorySaver로 컴파일 — 대화 상태가 메모리에 유지됨
memory_saver = MemorySaver()
graph = graph_builder.compile(checkpointer=memory_saver)

# thread_id로 대화 세션을 구분
config = {"configurable": {"thread_id": "session-1"}}

# 턴 1
result1 = graph.invoke(
    {"messages": [HumanMessage(content="제 이름은 홍길동입니다.")]},
    config=config,
)
print(f"[턴 1] {result1['messages'][-1].content}")
print()

[턴 1] 안녕하세요, 홍길동 님이시군요. 반갑습니다!



In [28]:
# 턴 2 — 같은 thread_id로 호출하면 이전 대화가 자동으로 포함됨
result2 = graph.invoke(
    {"messages": [HumanMessage(content="제 이름이 뭐였죠?")]},
    config=config,
)
print(f"[턴 2] {result2['messages'][-1].content}")
print(f"전체 메시지 수: {len(result2['messages'])}")

[턴 2] 네, 홍길동 님이십니다.
전체 메시지 수: 4


### thread_id로 세션 분리

`thread_id`를 다르게 하면 별도의 대화 세션으로 취급됩니다.
여러 사용자의 대화를 동시에 관리할 수 있습니다.

아래 코드는 두 개의 다른 세션을 보여줍니다.

In [29]:
# 세션 2 — 다른 thread_id
config2 = {"configurable": {"thread_id": "session-2"}}

result_s2 = graph.invoke(
    {"messages": [HumanMessage(content="제 이름은 김영희입니다.")]},
    config=config2,
)
print(f"[세션 2] {result_s2['messages'][-1].content}")
print()

# 세션 1에서 이름 확인 — 여전히 홍길동
result_s1 = graph.invoke(
    {"messages": [HumanMessage(content="제 이름이 뭐였죠?")]},
    config=config,  # session-1
)
print(f"[세션 1] {result_s1['messages'][-1].content}")
print()

# 세션 2에서 이름 확인 — 김영희
result_s2_check = graph.invoke(
    {"messages": [HumanMessage(content="제 이름이 뭐였죠?")]},
    config=config2,  # session-2
)
print(f"[세션 2] {result_s2_check['messages'][-1].content}")

[세션 2] 네, 김영희 님, 반갑습니다.

[세션 1] 네, 홍길동 님이십니다.

[세션 2] 네, 김영희 님이십니다.


### SqliteSaver로 영구 저장

`MemorySaver`는 프로세스 종료 시 사라집니다.
`SqliteSaver`를 사용하면 대화 상태를 SQLite 파일에 영구 저장할 수 있습니다.

아래 코드는 SqliteSaver를 사용하여 상태를 파일에 저장하는 예시를 보여줍니다.

In [33]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# SQLite 파일에 상태 저장
sqlite_conn = sqlite3.connect("chat_history.db", check_same_thread=False)
sqlite_saver = SqliteSaver(sqlite_conn)

# 같은 그래프를 SqliteSaver로 컴파일
graph_sqlite = graph_builder.compile(checkpointer=sqlite_saver)

config_sqlite = {"configurable": {"thread_id": "sqlite-session-1"}}

# 턴 1
result = graph_sqlite.invoke(
    {"messages": [HumanMessage(content="제 이름은 최민준입니다.")]},
    config=config_sqlite,
)
print(f"[턴 1] {result['messages'][-1].content}")

[턴 1] 안녕하세요, 최민준님! 만나서 반갑습니다.


In [34]:
# 턴 2 — SQLite에 저장된 이력이 자동으로 로드됨
result2 = graph_sqlite.invoke(
    {"messages": [HumanMessage(content="제 이름이 뭐였죠?")]},
    config=config_sqlite,
)
print(f"[턴 2] {result2['messages'][-1].content}")

[턴 2] 최민준 님이셨습니다.


In [35]:
# SQLite 파일이 생성되었는지 확인
import os
db_size = os.path.getsize("chat_history.db")
print(f"chat_history.db 파일 크기: {db_size:,} bytes")
print("(프로세스를 재시작해도 이 파일에 대화 이력이 남아있습니다)")

chat_history.db 파일 크기: 4,096 bytes
(프로세스를 재시작해도 이 파일에 대화 이력이 남아있습니다)


### MemorySaver vs SqliteSaver 정리

| 구분 | MemorySaver | SqliteSaver |
|------|------------|-------------|
| 저장 위치 | 메모리 (RAM) | SQLite 파일 |
| 영속성 | 프로세스 종료 시 소멸 | 파일로 영구 저장 |
| 속도 | 매우 빠름 | 빠름 |
| 용도 | 프로토타입, 테스트 | 단일 서버 운영 |
| 설정 복잡도 | 없음 | DB 연결 설정 필요 |

> 핵심: LangGraph는 Checkpointer만 교체하면 저장소를 바꿀 수 있습니다.
> 그래프 코드는 동일하게 유지됩니다.
> 프로토타입에서는 MemorySaver, 운영에서는 SqliteSaver(또는 PostgreSQL)를 사용합니다.

In [36]:
# 정리: SQLite 연결 닫기 + 파일 삭제
sqlite_conn.close()
if os.path.exists("chat_history.db"):
    os.remove("chat_history.db")
    print("chat_history.db 삭제 완료")

chat_history.db 삭제 완료


---

### 생각해보기

1. 멀티턴 대화에서 매번 전체 이력을 전송한다면, 100턴 대화의 마지막 호출에서는 입력 토큰이 얼마나 될까요? 이 비용 문제를 어떻게 해결할 수 있을까요?
2. `thread_id`로 세션을 분리한다면, 한 사용자가 여러 대화를 동시에 진행하는 경우 thread_id를 어떻게 설계해야 할까요?
3. InMemory 저장소와 SQLite 사이에 Redis를 선택해야 하는 상황은 어떤 경우일까요?

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 3-1: google-genai로 멀티턴 대화 구현 (contents 수동 관리)

google-genai의 `contents` 리스트를 수동으로 관리하여 3턴 대화를 구현하세요.

**요구사항**
1. `contents` 리스트 생성
2. system_instruction: "당신은 수학 선생님입니다. 풀이 과정을 간결하게 보여주세요."
3. 턴 1: "3 + 5는?"
4. 턴 2: "거기에 2를 곱하면?"
5. 턴 3: "처음 답에서 1을 빼면?"
6. 각 턴의 응답을 출력하고, 맥락이 유지되는지 확인

In [37]:
from google.genai.types import Content, Part

system_instruction = "당신은 수학 선생님입니다. 풀이 과정을 간결하게 보여주세요."

# TODO 1: contents 리스트 생성
contents = []  # 이 리스트를 사용하세요

questions = ["3 + 5는?", "거기에 2를 곱하면?", "처음 답에서 1을 빼면?"]

# TODO 2: 각 질문에 대해 반복하며 멀티턴 대화 구현
# 힌트: contents에 user Content 추가 → generate_content 호출 → model Content 추가

# ---------- 정답 ----------
for i, q in enumerate(questions, 1):
    contents.append(Content(role="user", parts=[Part(text=q)]))
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        config={"system_instruction": system_instruction},
        contents=contents,
    )
    contents.append(Content(role="model", parts=[Part(text=resp.text)]))
    print(f"[턴 {i}] Q: {q}")
    print(f"         A: {resp.text}")
    print()

# 검증
if len(contents) == 0:
    print("TODO를 완성해주세요")
else:
    print(f"총 contents 수: {len(contents)}개 (user 3 + model 3 = 6)")

[턴 1] Q: 3 + 5는?
         A: 8

[턴 2] Q: 거기에 2를 곱하면?
         A: $8 \times 2 = 16$

[턴 3] Q: 처음 답에서 1을 빼면?
         A: $8 - 1 = 7$

총 contents 수: 6개 (user 3 + model 3 = 6)


### 실습 3-2: google-genai client.chats.create() 사용

`client.chats.create()`로 채팅 세션을 생성하고 멀티턴 대화를 구현하세요.

**요구사항**
1. `client.chats.create()`로 세션 생성 (model: `gemini-2.5-flash`)
2. system_instruction: "당신은 여행 가이드입니다. 간결하게 답변하세요."
3. `send_message()`로 3턴 대화:
   - 턴 1: "제주도 추천 관광지는?"
   - 턴 2: "그 중 가족 여행에 좋은 곳은?"
   - 턴 3: "거기 근처 맛집도 알려주세요"
4. 각 턴의 응답을 출력

In [38]:
# TODO 1: 채팅 세션 생성
chat_session = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
chat_session = client.chats.create(
    model="gemini-2.5-flash",
    config={"system_instruction": "당신은 여행 가이드입니다. 간결하게 답변하세요."},
)

questions = [
    "제주도 추천 관광지는?",
    "그 중 가족 여행에 좋은 곳은?",
    "거기 근처 맛집도 알려주세요",
]

# TODO 2: send_message로 3턴 대화

# ---------- 정답 ----------
for i, q in enumerate(questions, 1):
    resp = chat_session.send_message(q)
    print(f"[턴 {i}] Q: {q}")
    print(f"         A: {resp.text}")
    print()

if chat_session is None:
    print("TODO를 완성해주세요")

[턴 1] Q: 제주도 추천 관광지는?
         A: 제주도 추천 관광지입니다:

*   **성산일출봉:** 유네스코 세계자연유산, 장엄한 일출 명소.
*   **한라산:** 제주의 상징, 다양한 등반 코스와 빼어난 자연경관.
*   **오름:** 새별오름, 용눈이오름 등 독특한 풍경의 화산체.
*   **협재해변, 월정리해변:** 아름다운 에메랄드빛 바다와 감성적인 카페들.
*   **사려니숲길, 비자림:** 피톤치드 가득한 힐링 숲길.
*   **천지연폭포, 정방폭포:** 웅장한 폭포의 절경.
*   **우도:** 에메랄드빛 바다와 아름다운 풍경의 섬 속의 섬.
*   **주상절리대:** 자연이 빚은 신비로운 돌기둥.
*   **동문시장:** 제주 특산물과 먹거리를 즐길 수 있는 재래시장.

[턴 2] Q: 그 중 가족 여행에 좋은 곳은?
         A: 가족 여행에 특히 좋은 곳들을 추천해 드립니다:

*   **협재해변, 월정리해변:** 아이들과 모래놀이, 물놀이하기 좋습니다. 주변에 편의시설과 카페도 많습니다.
*   **우도:** 페리를 타고 들어가 섬을 둘러보는 재미가 있고, 아름다운 해변과 다양한 먹거리가 있어 하루 종일 즐거운 시간을 보낼 수 있습니다.
*   **동문시장:** 다양한 먹거리(야시장 포함)와 기념품 구경으로 아이들도 흥미를 느낄 수 있는 활기찬 시장입니다.
*   **천지연폭포:** 잘 정비된 산책로를 따라 유모차 이동도 편리하며, 웅장한 폭포를 감상할 수 있습니다.
*   **사려니숲길, 비자림:** 피톤치드 가득한 숲길을 따라 가볍게 산책하며 자연을 느끼기 좋습니다.
*   **성산일출봉:** 유네스코 세계자연유산으로, 가벼운 오르막길을 따라 정상에 오르면 멋진 경치를 감상할 수 있습니다.
*   **오름 (특히 용눈이오름):** 완만한 경사로 아이들과 함께 오르기 좋은 오름들이 많습니다. (예: 용눈이오름은 비교적 평탄한 산책 코스)

[턴 3] Q: 거기 근처 맛집도 알려주세요
         A: 가족 여행

### 실습 3-3: LangChain InMemoryChatMessageHistory로 대화 관리

`InMemoryChatMessageHistory`를 사용하여 대화를 관리하세요.

**요구사항**
1. `InMemoryChatMessageHistory` 인스턴스 생성
2. SystemMessage 추가: "당신은 파이썬 튜터입니다. 초보자가 이해할 수 있게 설명하세요."
3. 대화 함수 `ask(user_input, history)` 구현:
   - history에 HumanMessage 추가
   - model.invoke(history.messages) 호출
   - history에 AI 응답 추가
   - 응답 문자열 반환
4. 3턴 대화 실행:
   - "리스트와 튜플의 차이는?"
   - "그러면 어떤 상황에 튜플을 쓰나요?"
   - "예시 코드를 보여주세요"

In [39]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, SystemMessage

# TODO 1: InMemoryChatMessageHistory 생성 + SystemMessage 추가
chat_history = None  # 이 줄을 수정하세요

# TODO 2: 대화 함수 구현
def ask(user_input: str, history) -> str:
    pass  # 이 함수를 구현하세요

# ---------- 정답 ----------
chat_history = InMemoryChatMessageHistory()
chat_history.add_message(
    SystemMessage(content="당신은 파이썬 튜터입니다. 초보자가 이해할 수 있게 설명하세요.")
)

def ask(user_input: str, history) -> str:
    history.add_message(HumanMessage(content=user_input))
    response = model.invoke(history.messages)
    history.add_message(response)
    return response.content

# 검증
questions = [
    "리스트와 튜플의 차이는?",
    "그러면 어떤 상황에 튜플을 쓰나요?",
    "예시 코드를 보여주세요",
]

if chat_history is not None:
    for i, q in enumerate(questions, 1):
        answer = ask(q, chat_history)
        print(f"[턴 {i}] Q: {q}")
        print(f"         A: {answer}")
        print()
    print(f"총 메시지 수: {len(chat_history.messages)}")
else:
    print("TODO를 완성해주세요")

[턴 1] Q: 리스트와 튜플의 차이는?
         A: 안녕하세요! 파이썬 튜터입니다. 😊

리스트(List)와 튜플(Tuple)은 파이썬에서 여러 개의 값을 순서대로 묶어서 저장할 때 사용하는 아주 유용한 도구들이에요. 겉으로 보기엔 비슷해 보여도, 아주 중요한 차이점이 하나 있답니다. 바로 **'수정 가능 여부'**예요.

이 차이점을 이해하기 위해 아주 간단한 비유를 들어볼게요.

---

### 1. 리스트 (List) - "수정 가능한 쇼핑 목록"

리스트는 마치 **"수정 가능한 쇼핑 목록"** 같아요.

*   **특징:**
    *   여러 개의 값을 순서대로 저장해요.
    *   가장 큰 특징은 한 번 만들고 나서 **안에 있는 값을 자유롭게 추가, 삭제, 변경**할 수 있다는 거예요.
    *   `대괄호 []`를 사용해서 만들어요.
*   **비유:**
    *   종이에 연필로 쓴 쇼핑 목록이라고 생각해보세요.
    *   "사과"를 적었다가 "배"로 바꿀 수도 있고, "우유"를 추가할 수도 있고, "빵"을 지울 수도 있죠?
    *   이렇게 내용물을 마음대로 바꿀 수 있는 것이 바로 리스트예요.

*   **예시 코드:**

    ```python
    # 리스트 만들기
    my_list = ["사과", "바나나", "체리"]
    print(f"원래 리스트: {my_list}") # 원래 리스트: ['사과', '바나나', '체리']

    # 값 추가하기 (append)
    my_list.append("오렌지")
    print(f"오렌지 추가 후: {my_list}") # 오렌지 추가 후: ['사과', '바나나', '체리', '오렌지']

    # 값 변경하기 (인덱스 사용)
    my_list[0] = "포도" # 첫 번째 값("사과")을 "포도"로 변경
    print(f"첫 번째 값 변경 후: {my_list}") # 첫 번째 값 변경 후: ['포도', '바나나', '체리', '오렌지'

### 실습 3-4: LangGraph MemorySaver로 멀티턴 대화

LangGraph의 `StateGraph`와 `MemorySaver`를 사용하여 멀티턴 대화를 구현하세요.

**요구사항**
1. `chat_node` 함수 정의: state에서 messages를 받아 model.invoke() 호출
2. `StateGraph(MessagesState)`로 그래프 정의
3. `MemorySaver`로 컴파일
4. thread_id: `"practice-session"`
5. 3턴 대화:
   - "제 취미는 독서입니다."
   - "제 취미가 뭐라고 했죠?"
   - "독서와 비슷한 다른 취미를 추천해주세요."

In [40]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

# TODO 1: chat_node 함수 정의
def chat_node(state: MessagesState):
    pass  # 이 함수를 구현하세요

# TODO 2: 그래프 정의 (StateGraph → add_node → add_edge)
builder = None  # 이 줄부터 수정하세요

# TODO 3: MemorySaver로 컴파일
practice_graph = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
def chat_node(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

practice_graph = builder.compile(checkpointer=MemorySaver())

# 검증
questions = [
    "제 취미는 독서입니다.",
    "제 취미가 뭐라고 했죠?",
    "독서와 비슷한 다른 취미를 추천해주세요.",
]

if practice_graph is not None:
    cfg = {"configurable": {"thread_id": "practice-session"}}
    for i, q in enumerate(questions, 1):
        result = practice_graph.invoke(
            {"messages": [HumanMessage(content=q)]},
            config=cfg,
        )
        print(f"[턴 {i}] Q: {q}")
        print(f"         A: {result['messages'][-1].content}")
        print()
else:
    print("TODO를 완성해주세요")

[턴 1] Q: 제 취미는 독서입니다.
         A: 아, 독서가 취미시군요! 정말 좋은 취미네요. 😊

마음을 풍요롭게 하고, 새로운 지식을 얻을 수 있는 훌륭한 활동이죠.

주로 어떤 종류의 책을 읽으시는 편이세요? 혹시 추천해주실 만한 책이 있으신가요?

[턴 2] Q: 제 취미가 뭐라고 했죠?
         A: 네, 독서라고 하셨습니다. 😊

[턴 3] Q: 독서와 비슷한 다른 취미를 추천해주세요.
         A: 독서는 지적인 자극, 스토리텔링, 상상력, 그리고 깊은 몰입을 제공하는 취미죠. 이런 독서의 매력과 비슷한 경험을 할 수 있는 다른 취미들을 추천해 드릴게요!

1.  **오디오북 (Audiobooks):**
    *   **유사점:** 가장 직접적인 대안입니다. 눈으로 글자를 읽는 대신 귀로 이야기를 듣는다는 점만 다를 뿐, 책의 내용을 그대로 즐길 수 있습니다. 이동 중이나 다른 활동을 하면서도 독서의 즐거움을 느낄 수 있죠.

2.  **팟캐스트 (Podcasts):**
    *   **유사점:** 다양한 주제의 팟캐스트가 있습니다. 스토리텔링 형식의 다큐멘터리, 역사, 과학, 시사, 심지어 소설을 연재하는 팟캐스트까지. 지적인 호기심을 충족시키거나 흥미로운 이야기를 듣는다는 점에서 독서와 매우 비슷합니다.

3.  **다큐멘터리 시청 및 비평 (Documentary Watching & Analysis):**
    *   **유사점:** 논픽션 독서와 비슷하게 특정 주제에 대한 깊이 있는 지식을 얻고 세상을 이해하는 시야를 넓힐 수 있습니다. 단순히 시청하는 것을 넘어, 내용을 분석하고 비판적으로 생각하는 과정은 책을 읽고 사색하는 것과 유사합니다.

4.  **글쓰기 (Writing - 일기, 블로그, 소설, 시 등):**
    *   **유사점:** 독서가 타인의 생각을 읽는 것이라면, 글쓰기는 자신의 생각을 정리하고 표현하는 것입니다. 상상력을 발휘하거나 특정 주제에 대해 깊이 탐구하는 과정이 독서와 상호 보

### 실습 3-5: thread_id로 세션 분리

동일한 LangGraph에서 thread_id를 다르게 하여 두 개의 독립적인 대화 세션을 운영하세요.

**요구사항**
1. 실습 3-4에서 만든 그래프를 재사용하거나 새로 만드세요
2. 세션 A (thread_id: `"user-alice"`):
   - 턴 1: "제 이름은 Alice입니다."
   - 턴 2: "제 이름이 뭐였죠?"
3. 세션 B (thread_id: `"user-bob"`):
   - 턴 1: "제 이름은 Bob입니다."
   - 턴 2: "제 이름이 뭐였죠?"
4. 각 세션이 독립적으로 동작하는지 확인

In [41]:
# TODO: 그래프 생성 (실습 3-4 재사용 또는 새로 만들기)
session_graph = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
def chat_fn(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

b = StateGraph(MessagesState)
b.add_node("chat", chat_fn)
b.add_edge(START, "chat")
b.add_edge("chat", END)
session_graph = b.compile(checkpointer=MemorySaver())

# 검증
if session_graph is not None:
    # 세션 A
    cfg_a = {"configurable": {"thread_id": "user-alice"}}
    r = session_graph.invoke({"messages": [HumanMessage(content="제 이름은 Alice입니다.")]}, config=cfg_a)
    print(f"[Alice 턴1] {r['messages'][-1].content}")

    # 세션 B
    cfg_b = {"configurable": {"thread_id": "user-bob"}}
    r = session_graph.invoke({"messages": [HumanMessage(content="제 이름은 Bob입니다.")]}, config=cfg_b)
    print(f"[Bob   턴1] {r['messages'][-1].content}")
    print()

    # 각 세션에서 이름 확인
    r_a = session_graph.invoke({"messages": [HumanMessage(content="제 이름이 뭐였죠?")]}, config=cfg_a)
    print(f"[Alice 턴2] {r_a['messages'][-1].content}")

    r_b = session_graph.invoke({"messages": [HumanMessage(content="제 이름이 뭐였죠?")]}, config=cfg_b)
    print(f"[Bob   턴2] {r_b['messages'][-1].content}")
else:
    print("TODO를 완성해주세요")

[Alice 턴1] 네, Alice 님이시군요. 만나서 반갑습니다!
[Bob   턴1] 안녕하세요, Bob님! 만나서 반갑습니다.

[Alice 턴2] Alice님이십니다.
[Bob   턴2] Bob님, 당신의 이름은 **Bob**입니다.


### 실습 3-6: SqliteSaver 적용

실습 3-4의 그래프를 `SqliteSaver`로 교체하여 영구 저장이 되는지 확인하세요.

**요구사항**
1. `sqlite3.connect()`로 DB 연결 (파일명: `"practice_chat.db"`)
2. `SqliteSaver`로 그래프 컴파일
3. 1턴 대화 실행 후 DB 파일 크기 확인
4. 정리: DB 연결 닫기 + 파일 삭제

In [42]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# TODO 1: SQLite 연결 생성
conn = None  # 이 줄을 수정하세요

# TODO 2: SqliteSaver로 그래프 컴파일
sqlite_graph = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
conn = sqlite3.connect("practice_chat.db", check_same_thread=False)

def chat_fn2(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

b2 = StateGraph(MessagesState)
b2.add_node("chat", chat_fn2)
b2.add_edge(START, "chat")
b2.add_edge("chat", END)
sqlite_graph = b2.compile(checkpointer=SqliteSaver(conn))

# 검증
if sqlite_graph is not None:
    cfg = {"configurable": {"thread_id": "sqlite-practice"}}
    result = sqlite_graph.invoke(
        {"messages": [HumanMessage(content="안녕하세요! 테스트입니다.")]},
        config=cfg,
    )
    print(f"응답: {result['messages'][-1].content}")
    print(f"DB 파일 크기: {os.path.getsize('practice_chat.db'):,} bytes")

    # 정리
    conn.close()
    os.remove("practice_chat.db")
    print("정리 완료")
else:
    print("TODO를 완성해주세요")

응답: 안녕하세요! 만나서 반갑습니다. 테스트 성공입니다! 😊
DB 파일 크기: 4,096 bytes
정리 완료


---

### 생각해보기

1. google-genai의 `client.chats.create()`와 LangGraph의 `MemorySaver`는 둘 다 메모리 기반입니다. 어떤 상황에서 어떤 것을 선택하겠습니까?
2. 실습 3-5에서 세션 A와 B가 분리되었습니다. 만약 두 세션 간에 정보를 공유해야 한다면 어떻게 설계하겠습니까?
3. SqliteSaver는 파일 기반이라 단일 서버에서만 동작합니다. 서버가 2대 이상인 환경에서는 어떤 저장소를 선택해야 할까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 3-1: 대화 맥락을 유지하는 간단한 Q&A 봇 구현 (난이도: ★★☆)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

특정 주제에 대해 대화 맥락을 유지하면서 답변하는 Q&A 봇을 구현하세요.

**요구사항**
- LangGraph StateGraph + MemorySaver 사용
- System Prompt로 봇의 역할 정의 (주제는 자유 선택)
- 최소 5턴 대화를 진행하며, 이전 맥락을 참조하는 질문을 포함할 것
- 각 턴의 입력 토큰 수를 출력하여 누적 비용을 관찰할 것

**평가 기준**
- 5턴 모두 이전 맥락을 올바르게 참조하는가?
- system prompt가 잘 설계되어 일관된 답변을 하는가?
- 토큰 누적을 관찰하고 분석할 수 있는가?

**힌트**
- 3번째, 5번째 턴에서 "아까 말한", "처음에 말한" 같은 맥락 참조 질문을 넣어보세요
- `usage_metadata`에서 토큰 수를 추출하려면 AIMessage의 속성을 확인하세요
- system prompt에 답변 길이 제약을 넣으면 토큰 누적을 줄일 수 있습니다

In [43]:
# 1단계: System Prompt + 그래프 구성
# 여기에 코드를 작성하세요

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import MemorySaver

# LLM 초기화
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_API_KEY,
)

# System Prompt 정의 - 파이썬 튜터 봇
SYSTEM_PROMPT = """당신은 Python 입문자를 위한 친절한 튜터 봇입니다.

역할:
- Python 기초 문법, 자료구조, 함수, 클래스 등을 쉽게 설명합니다.
- 비유와 예제 코드를 적극 활용합니다.
- 이전 대화에서 다룬 내용을 참조하여 연결성 있는 설명을 합니다.

규칙:
- 답변은 5문장 이내로 간결하게 합니다.
- 코드 예제는 3줄 이내로 짧게 유지합니다.
- 모르는 내용은 솔직히 모른다고 합니다.
"""

# 챗봇 노드 정의
def chatbot(state: MessagesState):
    messages = state["messages"]
    if not messages or not isinstance(messages[0], SystemMessage):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages

    response = model.invoke(messages)
    return {"messages": [response]}

# 그래프 구성
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

# MemorySaver로 대화 상태 유지
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("그래프 구성 완료")

그래프 구성 완료


In [44]:
# 2단계: 5턴 대화 실행 + 토큰 누적 관찰
# 여기에 코드를 작성하세요

conversations = [
    "파이썬에서 리스트와 튜플의 차이가 뭐야?",
    "리스트에 요소를 추가하는 방법을 알려줘",
    "아까 말한 튜플은 요소 추가가 안 된다고 했잖아, 그러면 튜플은 언제 쓰는 게 좋아?",
    "딕셔너리는 리스트랑 어떻게 달라?",
    "처음부터 지금까지 배운 자료구조들을 한 줄씩 정리해줘",
]

# thread_id로 대화 세션 구분
config = {"configurable": {"thread_id": "python-tutor-session-1"}}

# 토큰 누적 추적용
token_log = []
cumulative_input = 0
cumulative_output = 0

for i, user_input in enumerate(conversations, 1):
    print(f"\n{'='*60}")
    print(f"🧑 [{i}턴] {user_input}")
    print(f"{'='*60}")

    result = graph.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
    )

    ai_message = result["messages"][-1]
    print(f"\n🤖 {ai_message.content}")

    # 토큰 사용량 추출
    usage = ai_message.usage_metadata
    if usage:
        input_tokens = usage.get("input_tokens", 0)
        output_tokens = usage.get("output_tokens", 0)
        cumulative_input += input_tokens
        cumulative_output += output_tokens

        token_log.append({
            "turn": i,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cumulative_input": cumulative_input,
            "cumulative_output": cumulative_output,
        })

        print(f"\n이번 턴 - 입력: {input_tokens} | 출력: {output_tokens}")
        print(f"누적   - 입력: {cumulative_input} | 출력: {cumulative_output}")


🧑 [1턴] 파이썬에서 리스트와 튜플의 차이가 뭐야?

🤖 안녕하세요! 파이썬의 리스트와 튜플은 여러 값을 담는 자료형이지만, 가장 큰 차이는 '변경 가능성'에 있어요. 📝

리스트는 `[]` 대괄호를 사용하고, 한 번 만들고 나서도 값을 추가하거나 삭제하는 등 자유롭게 '변경'할 수 있습니다. 마치 쇼핑 목록처럼 언제든 수정할 수 있죠.

반면 튜플은 `()` 소괄호를 사용하며, 한 번 만들면 값을 절대 바꿀 수 없는 '변경 불가능한' 자료형이에요. 생일이나 좌표처럼 고정된 데이터를 저장할 때 주로 사용해요.

```python
# 리스트는 변경 가능
my_list = [1, 2]
my_list.append(3) # [1, 2, 3]

# 튜플은 변경 불가능
my_tuple = (1, 2)
# my_tuple.append(3) # 에러 발생!
```
이렇게 변경이 필요한 데이터는 리스트를, 바뀌지 않아야 할 데이터는 튜플을 사용한다고 기억하면 쉬울 거예요!

이번 턴 - 입력: 139 | 출력: 1170
누적   - 입력: 139 | 출력: 1170

🧑 [2턴] 리스트에 요소를 추가하는 방법을 알려줘

🤖 네, 리스트에 요소를 추가하는 가장 흔한 방법은 `append()`와 `insert()` 두 가지가 있어요. 😊

1.  **`append()`**: 리스트의 **맨 뒤**에 새로운 요소를 추가할 때 사용해요. 가장 많이 쓰이는 방법이랍니다.
2.  **`insert(인덱스, 요소)**: 리스트의 **원하는 위치(인덱스)**에 요소를 삽입할 때 사용해요.

예시 코드를 보면 더 쉽게 이해할 수 있을 거예요!

```python
fruits = ["사과", "바나나"]

fruits.append("오렌지") # 맨 뒤에 추가
# fruits는 이제 ["사과", "바나나", "오렌지"]

fruits.insert(1, "포도") # 인덱스 1 위치에 "포도" 삽입
# fruits는 이제 ["사과", "포도", "바나나", "오렌지"]
```
이렇게 `app

In [46]:
# 3단계: 결과 분석
# 토큰 누적 추이, 맥락 유지 여부 등을 정리하세요

import pandas as pd

df = pd.DataFrame(token_log)
print("=" * 60)
print("턴별 토큰 사용량 분석")
print("=" * 60)
print(df.to_string(index=False))

print(f"\n{'='*60}")
print("분석 요약")
print(f"{'='*60}")
print(f"총 입력 토큰: {cumulative_input}")
print(f"총 출력 토큰: {cumulative_output}")
print(f"총 토큰:     {cumulative_input + cumulative_output}")

if len(df) >= 2:
    first_input = df.iloc[0]["input_tokens"]
    last_input = df.iloc[-1]["input_tokens"]
    growth = last_input / first_input if first_input > 0 else 0
    print(f"\n1턴 입력 토큰: {int(first_input)}")
    print(f"5턴 입력 토큰: {int(last_input)}")
    print(f"입력 토큰 증가율: {growth:.1f}배")

턴별 토큰 사용량 분석
 turn  input_tokens  output_tokens  cumulative_input  cumulative_output
    1           139           1170               139               1170
    2           390            448               529               1618
    3           658            931              1187               2549
    4           908            960              2095               3509
    5          1220            356              3315               3865

분석 요약
총 입력 토큰: 3315
총 출력 토큰: 3865
총 토큰:     7180

1턴 입력 토큰: 139
5턴 입력 토큰: 1220
입력 토큰 증가율: 8.8배


---

### 생각해보기

1. 5턴 대화에서 입력 토큰 수가 어떤 패턴으로 증가했나요? 10턴, 50턴이 되면 비용이 어떻게 될지 예측해보세요.
2. 지금 구현한 봇에서 가장 먼저 개선해야 할 점은 무엇인가요? (힌트: 노트북 7의 컨텍스트 관리 전략을 미리 생각해보세요)
3. 실제 서비스에서 이 봇을 배포한다면, MemorySaver 대신 어떤 저장소를 선택하겠습니까? 그 이유는?